# Lesson 1 — Introduction to Qiskit and the Quantum Stack

**Quantum Computing with Qiskit**

This notebook accompanies Lesson 1. Work through the cells in order. Every cell is meant to be run; modify the code freely and re-run to experiment.

**Topics covered:**
- Installing Qiskit in Colab
- Building and drawing your first quantum circuit
- Running on a local simulator with `AerSimulator`
- Connecting to IBM Quantum (optional)
- Exploring the transpilation pipeline

## 1. Installation

Run this cell once at the start of each Colab session. It installs the three Qiskit packages used throughout this course.

In [ ]:
!pip install qiskit qiskit-aer qiskit-ibm-runtime --quiet

In [ ]:
import qiskit
import qiskit_aer
import qiskit_ibm_runtime

print("Qiskit:             ", qiskit.__version__)
print("Qiskit Aer:         ", qiskit_aer.__version__)
print("Qiskit IBM Runtime: ", qiskit_ibm_runtime.__version__)

## 2. Your First Quantum Circuit

A `QuantumCircuit` is built by creating the circuit object and then calling gate methods on it.

The circuit below creates a **Bell state**: an entangled two-qubit state where measuring one qubit instantly determines the other.

- `h(0)` applies a Hadamard gate to qubit 0, putting it in superposition.
- `cx(0, 1)` applies a CNOT gate: qubit 0 controls qubit 1.
- `measure([0, 1], [0, 1])` maps qubits to classical bits for readout.

In [ ]:
from qiskit import QuantumCircuit

qc = QuantumCircuit(2, 2)   # 2 qubits, 2 classical bits
qc.h(0)                     # Hadamard on qubit 0
qc.cx(0, 1)                 # CNOT: qubit 0 controls qubit 1
qc.measure([0, 1], [0, 1])  # measure both qubits

print(qc)  # text drawing

In [ ]:
# Graphical drawing (requires matplotlib, which Colab provides)
qc.draw('mpl')

### Inspecting the circuit

Before running, you can check some properties of the circuit.

In [ ]:
print("Number of qubits:        ", qc.num_qubits)
print("Number of classical bits:", qc.num_clbits)
print("Circuit depth:           ", qc.depth())
print("Gate counts:             ", qc.count_ops())

## 3. Running on a Local Simulator

`AerSimulator` runs circuits on your CPU: no internet, no IBM account, no queue.

In [ ]:
from qiskit import transpile
from qiskit_aer import AerSimulator

# Create the simulator
simulator = AerSimulator()

# Always transpile before running
transpiled = transpile(qc, simulator)

# Run for 1024 shots (1024 independent measurements)
job = simulator.run(transpiled, shots=1024)

# Retrieve results
result = job.result()
counts = result.get_counts()
print("Counts:", counts)

In [ ]:
from qiskit.visualization import plot_histogram

plot_histogram(counts, title='Bell State Measurement Results')

**Expected output:** Two bars of roughly equal height at `'00'` and `'11'`.

The outcomes `'01'` and `'10'` should not appear (or appear very rarely due to floating-point rounding). This is the signature of entanglement: the two qubits are always in the same state when measured.

### Effect of shot count

More shots give a better estimate of the true probability distribution. Run the cell below to see how the histogram stabilizes as shots increase.

In [ ]:
import matplotlib.pyplot as plt
from qiskit.visualization import plot_histogram

shot_counts = [10, 100, 1000, 10000]
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for ax, shots in zip(axes, shot_counts):
    job = simulator.run(transpiled, shots=shots)
    counts_n = job.result().get_counts()
    plot_histogram(counts_n, ax=ax, title=f"{shots} shots")

plt.suptitle("Convergence with Shot Count", fontsize=13)
plt.tight_layout()
plt.show()

## 4. IBM Quantum Account Setup (Optional)

This section requires an IBM Quantum account and API token. If you do not have one yet, skip to Section 5 and return here when you are ready.

**How to get your token:**
1. Go to [quantum.ibm.com](https://quantum.ibm.com) and log in.
2. Click your profile icon and select **Manage account**.
3. Under **API token**, copy your token.

**Recommended:** Store your token as a Colab secret (key icon in left sidebar) with the name `IBM_TOKEN`, then load it as shown below.

In [ ]:
# Option A: load from Colab secrets (recommended)
from google.colab import userdata
from qiskit_ibm_runtime import QiskitRuntimeService

try:
    token = userdata.get("IBM_TOKEN")
    QiskitRuntimeService.save_account(
        channel="ibm_quantum_platform",
        token=token,
        overwrite=True
    )
    print("Account saved successfully.")
except Exception as e:
    print("Could not load IBM_TOKEN from secrets:", e)
    print("Use Option B below instead.")

In [ ]:
# Option B: paste your token directly (less secure; do not share this notebook)
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum",
#     token="PASTE_YOUR_TOKEN_HERE",
#     overwrite=True
# )

In [ ]:
# Load the service and list available backends
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backends = service.backends(operational=True)

print(f"{'Backend':<35} {'Qubits':>6} {'Pending jobs':>12}")
print("-" * 56)
for b in backends:
    try:
        pending = b.status().pending_jobs
    except Exception:
        pending = "n/a"
    print(f"{b.name:<35} {b.num_qubits:>6} {str(pending):>12}")

In [ ]:
# Select the least-busy real hardware backend
backend = service.least_busy(operational=True, simulator=False)
print("Selected backend:", backend.name)
print("Number of qubits:", backend.num_qubits)

To run on real hardware, use the same `transpile` and `run` pattern as the simulator. Jobs may take several minutes to complete depending on queue depth.

```python
transpiled_hw = transpile(qc, backend, optimization_level=2)
job_hw = backend.run(transpiled_hw, shots=1024)
print("Job ID:", job_hw.job_id())  # save this to retrieve results later
```

We will submit real hardware jobs starting in Lesson 8. For now, the local simulator is sufficient for all exercises.

## 5. Exploring the Transpilation Pipeline

Transpilation converts your abstract circuit to a form the backend can execute. The two main constraints it handles are:
- **Basis gates:** real hardware supports only a small fixed gate set.
- **Connectivity:** only adjacent qubits can interact; SWAP gates are inserted where needed.

We use `FakeBrisbane`, a local mock of a real IBM device, to explore transpilation without an IBM account.

In [ ]:
from qiskit_ibm_runtime.fake_provider import FakeBrisbane

fake_backend = FakeBrisbane()

print("Basis gates:", fake_backend.operation_names)
print("Number of qubits:", fake_backend.num_qubits)

In [ ]:
# Compare circuit depth at different optimization levels
from qiskit import transpile

for level in range(4):
    t = transpile(qc, fake_backend, optimization_level=level)
    print(f"Level {level}: depth={t.depth():3d}  gates={dict(t.count_ops())}")

In [ ]:
# Draw the original and transpiled circuits side by side
import matplotlib.pyplot as plt

t3 = transpile(qc, fake_backend, optimization_level=3)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
qc.draw('mpl', ax=axes[0])
axes[0].set_title('Original circuit')
t3.draw('mpl', ax=axes[1], fold=-1)
axes[1].set_title('After transpilation (level 3)')
plt.tight_layout()
plt.show()

In [ ]:
# Visualize the device connectivity (coupling map)
from qiskit.visualization import plot_gate_map

plot_gate_map(fake_backend)

The coupling map shows which qubit pairs can interact directly. Gates between non-adjacent qubits require SWAP insertions, which increase circuit depth and add error.

## 6. Exercises

Work through these exercises to consolidate what you learned in this lesson.

### Exercise 1: A single-qubit circuit

Build a circuit with 1 qubit and 1 classical bit. Apply an X gate (Pauli-X, the quantum NOT gate) followed by a measurement. Run it on `AerSimulator` with 512 shots and print the counts.

Starting from $|0\rangle$, an X gate should always produce $|1\rangle$. What counts do you expect?

In [ ]:
# Exercise 1: your solution here
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

# Build the circuit
qc_ex1 = QuantumCircuit(1, 1)
# TODO: apply X gate and measurement

# Run and print counts
sim = AerSimulator()
t = transpile(qc_ex1, sim)
counts = sim.run(t, shots=512).result().get_counts()
print(counts)
qc_ex1.draw('mpl')

### Exercise 2: Shot noise

Build a single-qubit circuit that applies an H gate and then measures. With a perfect simulator, you expect 50% `'0'` and 50% `'1'`.

Run the circuit five times, each time with 100 shots. Print the fraction of `'0'` outcomes for each run. How much does it vary? Now repeat with 10,000 shots per run. How does the variance change?

In [ ]:
# Exercise 2: your solution here
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

qc_ex2 = QuantumCircuit(1, 1)
# TODO: apply H gate and measurement

sim = AerSimulator()
t = transpile(qc_ex2, sim)

# TODO: run 5 times with 100 shots each and print the fraction of '0' outcomes
# Then repeat with 10000 shots


### Exercise 3: Transpilation depth comparison

Build a three-qubit circuit that:
1. Applies H to qubit 0
2. Applies CNOT with control 0, target 1
3. Applies CNOT with control 1, target 2
4. Measures all three qubits

Transpile it for `FakeBrisbane` at all four optimization levels (0 through 3). For each level, print the circuit depth and the total gate count. Draw the transpiled circuit at level 3.

In [ ]:
# Exercise 3: your solution here
from qiskit import QuantumCircuit, transpile
from qiskit_ibm_runtime.fake_provider import FakeBrisbane

qc_ex3 = QuantumCircuit(3, 3)
# TODO: build the three-qubit GHZ-like circuit

fake = FakeBrisbane()
# TODO: transpile at levels 0-3 and compare depth and gate counts


### Exercise 4: Reading transpiler output

Take the Bell state circuit `qc` from Section 2. Transpile it for `FakeBrisbane` at optimization level 2. Then answer:

1. How many gates does the transpiled circuit have? Which gate types appear?
2. The original circuit used an `h` gate. Does the transpiled circuit still use `h`? If not, what replaced it and why?
3. Print `t.layout` to see which logical qubit was mapped to which physical qubit.

In [ ]:
# Exercise 4: your solution here
from qiskit import transpile
from qiskit_ibm_runtime.fake_provider import FakeBrisbane

fake = FakeBrisbane()
t = transpile(qc, fake, optimization_level=2)

# TODO: print gate counts, check for 'h' gate, print layout
print("Gate counts:", t.count_ops())
# ...


## Summary

In this lesson you:

- Installed Qiskit and its companion packages in Google Colab.
- Built a two-qubit Bell state circuit using `QuantumCircuit` and gate methods.
- Ran the circuit on `AerSimulator` with multiple shot counts and visualized the results with `plot_histogram`.
- Set up an IBM Quantum account and explored the available backends.
- Used `FakeBrisbane` to explore how the transpiler adapts a circuit to real hardware constraints.
- Compared circuit depth at different optimization levels and visualized the coupling map.

**Lesson 2** covers the full `QuantumCircuit` API, the matrix representations of single-qubit gates, multi-qubit gates, and statevector simulation.